# AutoFleet A1: Fleet Service Intelligence agent (notebook edition)

An AI agent that answers *"What is preventing our robotaxi fleet from achieving target operational readiness, and what actions would improve it?"*
using **tools over fleet data** instead of guessing. This notebook hosts the whole thing: data, KPI tools, the tool-calling loop, the eval, and an MCP client.

**How it fits together**
```
question -> LLM picks a tool -> Python runs it -> result goes back to the LLM -> answer (max 4 tool calls)
```
Runs in **Jupyter** or **Google Colab**. Needs an Anthropic API key for the real agent; without one it falls back to an offline scripted backend so you can still run every cell.

## 0. Setup

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

REPO_URL = "https://github.com/NITYA1J/24hr-learn.git"   # used only on Colab / fresh machines
BRANCH   = "claude/modest-johnson-belfm3"

def find_root():
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd() / "autofleet-a1"]:
        if (p / "tools" / "kpis.py").exists():
            return p
    return None

ROOT = find_root()
if ROOT is None:                                         # e.g. a brand-new Colab runtime
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "repo"], check=True)
    ROOT = Path("repo/autofleet-a1").resolve()

os.chdir(ROOT); sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "pydantic", "anthropic"], check=True)
print("project root:", ROOT)

In [ ]:
# Data is committed, but regenerate it if missing (deterministic, seed 42).
if not (ROOT / "data" / "work_orders.csv").exists():
    subprocess.run([sys.executable, "data/generate.py"], check=True)
import pandas as pd
print(pd.read_csv("data/work_orders.csv").shape, "work orders")

## 1. The KPI tools (plain pandas functions)
These are the "dishes" the agent can order. Call them directly first so you know what they return.

In [ ]:
from tools import kpis
from tools.playbooks import search_playbooks
from IPython.display import Markdown, display

print(json.dumps(kpis.fleet_readiness()["fleet"], indent=2))
pd.DataFrame(kpis.fleet_readiness()["sites"])

In [ ]:
pd.DataFrame(kpis.top_bottlenecks(n=5)["top"])

## 2. What the model actually sees
Each function's **docstring** becomes the tool description and its **type hints** become the JSON schema. Vague docstring = wrong tool choice.

In [ ]:
from agent.registry import tool_specs
specs = tool_specs()
print([s["name"] for s in specs])
print(json.dumps(specs[5], indent=2))      # top_bottlenecks

## 3. Choose the brain
- **Claude** (real agent): set `ANTHROPIC_API_KEY` in your environment (or Colab Secrets), or paste it when prompted.
- **Scripted** (offline): checks the loop, tools and scorer only. It is *not* a model and its scores mean nothing about an LLM.

Force offline with `AUTOFLEET_MODE=scripted`.

In [ ]:
from agent.loop import run_agent, ClaudeBackend, ScriptedBackend

MODEL = "claude-sonnet-5-5"          # change to compare models (Block 8)
QUESTIONS = [json.loads(l) for l in Path("eval/questions.jsonl").read_text().splitlines()]

mode = os.environ.get("AUTOFLEET_MODE", "")
key = os.environ.get("ANTHROPIC_API_KEY", "")
if not mode and not key:
    try:
        from google.colab import userdata          # Colab Secrets
        key = userdata.get("ANTHROPIC_API_KEY") or ""
    except Exception:
        pass
if not mode and not key:
    from getpass import getpass
    key = getpass("Anthropic API key (blank = offline scripted backend): ").strip()
if mode == "scripted" or not key:
    llm, BACKEND = ScriptedBackend(QUESTIONS), "scripted"
else:
    llm, BACKEND = ClaudeBackend(model=MODEL, api_key=key), f"claude:{MODEL}"
print("backend:", BACKEND)

## 4. Ask the central question

In [ ]:
CENTRAL = "What is preventing our robotaxi fleet from achieving target operational readiness, and what actions would improve it?"
out = run_agent(CENTRAL, llm, verbose=True)
display(Markdown(out["answer"]))

Check the numbers against the answer key (`data/truth.json`): sensor calibration about 35% of downtime, Site C worst with 2 night technicians, Region 2 falling about 1.6 points per week, Site A healthy.

In [ ]:
truth = json.loads(Path("data/truth.json").read_text())
{k: truth[k] for k in ["sensor_calibration_downtime_share", "region_readiness_slope_points_per_week", "site_mean_readiness_pct"]}

## 5. A reusable `ask()` helper
Use it for follow-ups. It prints the tool calls so you can see the agent's reasoning.

In [ ]:
def ask(question: str, show_calls: bool = True):
    r = run_agent(question, llm, verbose=show_calls)
    display(Markdown(r["answer"]))
    return r

_ = ask("Compare backlog aging at Site C versus Site A.")

## 6. Evaluate it (the 30-question eval set)
Scores four things: **tool choice**, **numeric accuracy**, **refusal accuracy**, and a **hallucination flag** (any number in the answer that no tool returned).
A real Claude run makes about 30 to 90 API calls.

In [ ]:
from eval import score
Path("eval/results").mkdir(exist_ok=True)
safe = BACKEND.replace(":", "_")
out_path = Path(f"eval/results/{safe}.jsonl")
with out_path.open("w") as f:
    for q in QUESTIONS:
        r = run_agent(q["question"], llm)
        f.write(json.dumps({"id": q["id"], **r}) + "\n")
score.main(str(out_path))
if BACKEND == "scripted":
    print("\n(scripted backend = pipeline check only, NOT a model score)")

## 7. Use the same tools over MCP from inside the notebook
`server.py` exposes the exact same functions over the Model Context Protocol, so Claude Desktop / Claude Code can use them too. Here the notebook acts as an MCP *client*.

In [ ]:
try:
    from mcp import ClientSession, StdioServerParameters, stdio_client
except ImportError:
    print("pip install 'mcp>=2' to run this cell")
else:
    params = StdioServerParameters(command=sys.executable, args=["server.py"], cwd=str(ROOT))
    # Jupyter's sys.stderr has no file descriptor, so give the server process a real file for its stderr.
    errlog = open(os.devnull, "w")
    async with stdio_client(params, errlog=errlog) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = (await session.list_tools()).tools
            print("MCP tools:", [t.name for t in tools])
            res = await session.call_tool("top_bottlenecks", {"n": 1})
            print(res.content[0].text)

## Next steps (see `LEARNING.md`)
1. Record the **baseline** (run section 6 with Claude) and note it in the README.
2. **Compare models**: change `MODEL`, re-run section 6, and compare the tables.
3. **Break a docstring** in `tools/kpis.py` and see which question types drop.
4. Add messy data (Block 8) and a **`simulate`** what-if tool (Block 9). Each new tool function appears here and over MCP automatically.